In [3]:
import os
import re
import json
import pickle
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.utils.class_weight import compute_class_weight

# =====================
# CONFIG
# =====================
SEED = 42

# Chọn 15 category có số lượng mẫu nhiều nhất
TOP_K = 15

# None = dùng toàn bộ dữ liệu trong mỗi category
# Có thể đặt ví dụ 5000 nếu muốn cân bằng / train nhanh hơn
MAX_SAMPLES_PER_CLASS = None

# TF-IDF config cho pipeline học máy
MAX_FEATURES = 50000
NGRAM_RANGE = (1, 2)
MIN_DF = 2
MAX_DF = 0.95

# =====================
# Đường dẫn Kaggle
# =====================
INPUT_PATH = "/kaggle/input/datasets/rmisra/news-category-dataset/News_Category_Dataset_v3.json"

OUTPUT_DIR = Path("/kaggle/working")
FEATURE_DIR = OUTPUT_DIR / "features_ml"
MODEL_DIR = OUTPUT_DIR / "models_ml"
RESULT_DIR = OUTPUT_DIR / "results_ml"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# =====================
# Reproducibility
# =====================
random.seed(SEED)
np.random.seed(SEED)

print("Input path:", INPUT_PATH)
print("Output dir:", OUTPUT_DIR)


Input path: /kaggle/input/datasets/rmisra/news-category-dataset/News_Category_Dataset_v3.json
Output dir: /kaggle/working


In [4]:
import kagglehub
import os
from pathlib import Path

print("Downloading pre-processed dataset from Kaggle...")
# Download latest version of the pre-processed dataset
downloaded_dataset_path = kagglehub.dataset_download("kimanaru/dataset-tien-xu-li")

print("Path to downloaded pre-processed dataset files:", downloaded_dataset_path)

# Update RESULT_DIR to point to this downloaded directory
# Assuming the pre-processed CSVs (train_ml_clean_text.csv, etc.) are directly in this directory
# Adjust this path if the CSVs are in a subfolder within the downloaded dataset.
RESULT_DIR = Path(downloaded_dataset_path)

print("RESULT_DIR has been updated to:", RESULT_DIR)

Path to downloaded pre-processed dataset files: /kaggle/input/datasets/kimanaru/dataset-tien-xu-li
RESULT_DIR has been updated to: /kaggle/input/datasets/kimanaru/dataset-tien-xu-li


## 1. Trích xuất đặc trưng và huấn luyện mô hình với TF-IDF

Trong phần này, chúng ta sẽ thực hiện các bước sau:
1.  Tải lại dữ liệu đã tiền xử lý từ các file CSV đã lưu.
2.  Khởi tạo và huấn luyện `TfidfVectorizer` để chuyển đổi văn bản thành các vector TF-IDF.
3.  Mã hóa các nhãn thể loại thành số.
4.  Huấn luyện mô hình `RandomForestClassifier` trên các đặc trưng TF-IDF.
5.  Đánh giá hiệu suất của mô hình trên tập validation và test.
6.  Lưu trữ mô hình và `TfidfVectorizer` đã huấn luyện.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Tải lại dữ liệu đã tiền xử lý
train_df = pd.read_csv(RESULT_DIR / "train_ml_clean_text.csv")
val_df = pd.read_csv(RESULT_DIR / "val_ml_clean_text.csv")
test_df = pd.read_csv(RESULT_DIR / "test_ml_clean_text.csv")

# Điền các giá trị NaN trong cột 'ml_clean_text' bằng chuỗi rỗng
train_df["ml_clean_text"] = train_df["ml_clean_text"].fillna("")
val_df["ml_clean_text"] = val_df["ml_clean_text"].fillna("")
test_df["ml_clean_text"] = test_df["ml_clean_text"].fillna("")

print("Train data loaded, shape:", train_df.shape)
print("Validation data loaded, shape:", val_df.shape)
print("Test data loaded, shape:", test_df.shape)

Train data loaded, shape: (103685, 2)
Validation data loaded, shape: (22218, 2)
Test data loaded, shape: (22219, 2)


In [6]:
# Khởi tạo TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    ngram_range=NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF
)

# Fit và transform tập huấn luyện
X_train_tfidf = tfidf_vectorizer.fit_transform(train_df["ml_clean_text"])
X_val_tfidf = tfidf_vectorizer.transform(val_df["ml_clean_text"])
X_test_tfidf = tfidf_vectorizer.transform(test_df["ml_clean_text"])

print("Shape of X_train_tfidf:", X_train_tfidf.shape)
print("Shape of X_val_tfidf:", X_val_tfidf.shape)
print("Shape of X_test_tfidf:", X_test_tfidf.shape)

# Lưu vectorizer
with open(FEATURE_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)
print("TF-IDF Vectorizer saved to:", FEATURE_DIR / "tfidf_vectorizer.pkl")

Shape of X_train_tfidf: (103685, 50000)
Shape of X_val_tfidf: (22218, 50000)
Shape of X_test_tfidf: (22219, 50000)
TF-IDF Vectorizer saved to: /kaggle/working/features_ml/tfidf_vectorizer.pkl


In [7]:
from sklearn.preprocessing import LabelEncoder

# Mã hóa nhãn
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(train_df["category"])
y_val_encoded = label_encoder.transform(val_df["category"])
y_test_encoded = label_encoder.transform(test_df["category"])

# Lưu LabelEncoder
with open(FEATURE_DIR / "label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)
print("Label Encoder saved to:", FEATURE_DIR / "label_encoder.pkl")

Label Encoder saved to: /kaggle/working/features_ml/label_encoder.pkl


In [8]:
print("=== Huấn luyện Random Forest với TF-IDF ===")

# Khởi tạo mô hình Random Forest
rf_tfidf_model = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)

# Huấn luyện mô hình
rf_tfidf_model.fit(X_train_tfidf, y_train_encoded)

# Lưu mô hình
with open(MODEL_DIR / "rf_tfidf_model.pkl", "wb") as f:
    pickle.dump(rf_tfidf_model, f)
print("Random Forest (TF-IDF) model saved to:", MODEL_DIR / "rf_tfidf_model.pkl")

=== Huấn luyện Random Forest với TF-IDF ===
Random Forest (TF-IDF) model saved to: /kaggle/working/models_ml/rf_tfidf_model.pkl


In [9]:
print("=== Đánh giá mô hình Random Forest (TF-IDF) ===")

# Đánh giá trên tập Validation
y_val_pred_tfidf = rf_tfidf_model.predict(X_val_tfidf)
print("\nClassification Report (Validation Set):")
print(classification_report(y_val_encoded, y_val_pred_tfidf, target_names=label_encoder.classes_))
print("Accuracy (Validation Set):", accuracy_score(y_val_encoded, y_val_pred_tfidf))

# Đánh giá trên tập Test
y_test_pred_tfidf = rf_tfidf_model.predict(X_test_tfidf)
print("\nClassification Report (Test Set):")
print(classification_report(y_test_encoded, y_test_pred_tfidf, target_names=label_encoder.classes_))
print("Accuracy (Test Set):", accuracy_score(y_test_encoded, y_test_pred_tfidf))

=== Đánh giá mô hình Random Forest (TF-IDF) ===

Classification Report (Validation Set):
                precision    recall  f1-score   support

  BLACK VOICES       0.69      0.27      0.38       688
      BUSINESS       0.69      0.38      0.49       899
        COMEDY       0.63      0.34      0.44       810
 ENTERTAINMENT       0.61      0.72      0.66      2604
  FOOD & DRINK       0.70      0.71      0.71       951
HEALTHY LIVING       0.42      0.23      0.30      1004
 HOME & LIVING       0.77      0.59      0.67       648
     PARENTING       0.57      0.65      0.61      1318
       PARENTS       0.52      0.19      0.28       593
      POLITICS       0.75      0.92      0.82      5340
  QUEER VOICES       0.89      0.60      0.71       952
        SPORTS       0.73      0.55      0.63       762
STYLE & BEAUTY       0.76      0.79      0.77      1472
        TRAVEL       0.74      0.69      0.72      1485
      WELLNESS       0.59      0.78      0.67      2692

      accurac

## 4. Điều chỉnh siêu tham số (Hyperparameter Tuning) cho Random Forest với TF-IDF

Chúng ta sẽ sử dụng `RandomizedSearchCV` để tìm kiếm các siêu tham số tối ưu cho mô hình Random Forest khi sử dụng đặc trưng TF-IDF.

In [10]:
from sklearn.model_selection import RandomizedSearchCV, ShuffleSplit
from sklearn.ensemble import RandomForestClassifier

SEED = 42 # Define SEED within this cell for robustness

print("=== Bắt đầu điều chỉnh siêu tham số cho Random Forest (TF-IDF) ===")

# Định nghĩa không gian tìm kiếm siêu tham số
param_dist = {
    'n_estimators': [50, 100, 200],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'bootstrap': [True, False]
}

# Khởi tạo mô hình Random Forest
rf_tfidf = RandomForestClassifier(random_state=SEED, n_jobs=-1)

cv_strategy = ShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)


random_search_tfidf = RandomizedSearchCV(
    estimator=rf_tfidf,
    param_distributions=param_dist,
    n_iter=10, # Tăng số lượng iter để tìm kiếm tốt hơn
    cv=cv_strategy, # Sử dụng chiến lược validation mới
    verbose=1,
    random_state=SEED,
    n_jobs=-1, # Sử dụng tất cả các CPU cores có sẵn
    scoring='accuracy'
)

# Thực hiện tìm kiếm
random_search_tfidf.fit(X_train_tfidf, y_train_encoded)

print("Các siêu tham số tốt nhất:", random_search_tfidf.best_params_)

# Lấy mô hình tốt nhất
best_rf_tfidf_model = random_search_tfidf.best_estimator_

# Lưu mô hình tốt nhất
with open(MODEL_DIR / "rf_tfidf_model_tuned.pkl", "wb") as f:
    pickle.dump(best_rf_tfidf_model, f)
print("Random Forest (TF-IDF) model đã điều chỉnh siêu tham số được lưu tại:", MODEL_DIR / "rf_tfidf_model_tuned.pkl")

=== Bắt đầu điều chỉnh siêu tham số cho Random Forest (TF-IDF) ===
Fitting 1 folds for each of 10 candidates, totalling 10 fits
Các siêu tham số tốt nhất: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'bootstrap': True}
Random Forest (TF-IDF) model đã điều chỉnh siêu tham số được lưu tại: /kaggle/working/models_ml/rf_tfidf_model_tuned.pkl


**Kết quả chi tiết của các bộ thử nghiệm RandomizedSearchCV**

In [11]:
print("=== Kết quả chi tiết của các bộ thử nghiệm RandomizedSearchCV ===")
results_df = pd.DataFrame(random_search_tfidf.cv_results_)
display(results_df.sort_values(by='rank_test_score', ascending=True).head())

=== Kết quả chi tiết của các bộ thử nghiệm RandomizedSearchCV ===


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_estimators,param_min_samples_split,param_min_samples_leaf,param_max_features,param_max_depth,param_bootstrap,params,split0_test_score,mean_test_score,std_test_score,rank_test_score
7,635.213399,0.0,5.685200,0.0,200,5,1,log2,None,True,"{'n_estimators': 200, 'min_samples_split': 5, ...",0.686261,0.686261,0.0,1
6,579.996731,0.0,3.530215,0.0,50,2,1,log2,None,False,"{'n_estimators': 50, 'min_samples_split': 2, '...",0.664802,0.664802,0.0,2
4,315.808169,0.0,4.720353,0.0,200,2,2,sqrt,None,True,"{'n_estimators': 200, 'min_samples_split': 2, ...",0.660028,0.660028,0.0,3
8,31.098995,0.0,2.702675,0.0,50,2,2,log2,None,True,"{'n_estimators': 50, 'min_samples_split': 2, '...",0.585427,0.585427,0.0,4
3,24.168355,0.0,0.772706,0.0,100,2,1,sqrt,20,False,"{'n_estimators': 100, 'min_samples_split': 2, ...",0.299368,0.299368,0.0,5


In [12]:
from sklearn.metrics import classification_report, accuracy_score

print("=== Đánh giá mô hình Random Forest (TF-IDF) đã điều chỉnh ===")

# Đánh giá trên tập Validation
y_val_pred_tfidf_tuned = best_rf_tfidf_model.predict(X_val_tfidf)
print("\nClassification Report (Validation Set - Tuned TF-IDF):")
print(classification_report(y_val_encoded, y_val_pred_tfidf_tuned, target_names=label_encoder.classes_))
print("Accuracy (Validation Set - Tuned TF-IDF):", accuracy_score(y_val_encoded, y_val_pred_tfidf_tuned))

# Đánh giá trên tập Test
y_test_pred_tfidf_tuned = best_rf_tfidf_model.predict(X_test_tfidf)
print("\nClassification Report (Test Set - Tuned TF-IDF):")
print(classification_report(y_test_encoded, y_test_pred_tfidf_tuned, target_names=label_encoder.classes_))
print("Accuracy (Test Set - Tuned TF-IDF):", accuracy_score(y_test_encoded, y_test_pred_tfidf_tuned))

=== Đánh giá mô hình Random Forest (TF-IDF) đã điều chỉnh ===

Classification Report (Validation Set - Tuned TF-IDF):
                precision    recall  f1-score   support

  BLACK VOICES       0.78      0.17      0.27       688
      BUSINESS       0.83      0.28      0.41       899
        COMEDY       0.75      0.27      0.39       810
 ENTERTAINMENT       0.64      0.81      0.71      2604
  FOOD & DRINK       0.75      0.78      0.76       951
HEALTHY LIVING       0.61      0.24      0.34      1004
 HOME & LIVING       0.90      0.58      0.71       648
     PARENTING       0.67      0.53      0.59      1318
       PARENTS       0.66      0.17      0.26       593
      POLITICS       0.67      0.97      0.79      5340
  QUEER VOICES       0.91      0.50      0.65       952
        SPORTS       0.78      0.59      0.67       762
STYLE & BEAUTY       0.89      0.78      0.83      1472
        TRAVEL       0.81      0.75      0.78      1485
      WELLNESS       0.61      0.82      